# A reference resolves by code

A consumer sends a currency reference with an outdated label and symbol.

Run in a fresh kernel using the repository's Python environment. The matching script is `02_reference_payload.py`.


## Prepare the model

These declarations supply the complete model needed by this case. Run this cell before the experiment.


In [1]:
from pydantic import Field

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


class ShopDomain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currencies accepted by the shop", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the shop's currency vocabulary."""

    symbol: str = Field(description="Currency symbol")

    EUR = CurrencyEnumEntity(code="eur", display_name="Euro", symbol="€")  # noqa: F821
    USD = CurrencyEnumEntity(code="usd", display_name="US dollar", symbol="$")  # noqa: F821


@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Associate an amount with its currency."""

    id: str = Field(description="Tag identifier")
    amount: float = Field(description="Price amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")


## Run the experiment

A consumer sends a currency reference with an outdated label and symbol.


In [2]:
tag = PriceTagCoreEntity.model_validate(
    {
        "id": "tag-1",
        "amount": 9.99,
        "currency": {"code": "eur", "display_name": "Old label", "properties": {"symbol": "?"}},
    }
)
print("Shared value:", tag.currency is CurrencyEnumEntity.EUR)
print(tag.currency.display_name, tag.currency.symbol)
assert tag.currency is CurrencyEnumEntity.EUR


Shared value: True
Euro €


## Explain the result

The code selects EUR. The payload label and properties do not update, override or get compared with the declaration. The value in memory has the current declared label and symbol.
